In [441]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import ElementClickInterceptedException
from selenium.common.exceptions import StaleElementReferenceException, NoSuchElementException
import time
from utils import writeJson, readJson
import os
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.keys import Keys
import json
import re
from bs4 import BeautifulSoup as soup
import datetime
from datetime import datetime as dt
from tqdm import tqdm

In [ ]:
def getPlayerStats(driver):
    data = {}
    tables = driver.find_elements(By.CLASS_NAME, 'stats_table')
    table = None
    for t in tables:
        if "scout_full" in t.get_attribute("id"):
            table = t
            break
    tags = ['<br>', '<strong>', '</strong>']
    type_stat = 'Standard Stats'
    data[type_stat] = {}
    for row in table.find_elements(By.TAG_NAME, 'tr'):
        th = row.find_element(By.TAG_NAME, 'th')
        #print(f'--{row.get_property("className")} -- {row.text}')
        if row.get_property("className") == "thead over_header thead":
            type_stat = th.text
            data[type_stat] = {}
            #print(type_stat)
        
        data_desc = th.get_attribute('data-tip')
        

        tds = row.find_elements(By.TAG_NAME, 'td')
        if len(tds) > 0 and th.text != '':
            value, perc = tds[0].text , tds[1].text
            #f'{th.text} ({data_desc})'
            for t in tags:
                if data_desc != None:
                    data_desc = data_desc.replace(t, ' ')
                
            data[type_stat][th.text] = {'description': data_desc, 'value': value, 'percentile': perc.strip()}

    return data

def getRoles(role):
    rr=[]
    if '(' in role:
        roles_split = role.split('(')
    else:
        roles_split = [role]

    for r in roles_split:
        if '-' in r:
            r_split = r.split('-')
            rr.append(r_split[0])
            rl = r_split[1]
            if rl[-1] == ')':
                rl = rl[:-1]
            rr.append(rl)

        elif ',' in r:
            rr.append(r.split(',')[0])
        elif ')' in r:
            rr.append(r[:-1])
        else:
            rr.append(r.strip())


    return rr

def initializeDriver():
    driver = webdriver.Chrome()
    url = 'https://fbref.com/en/'
    driver.get(url)
    cookie_button = driver.find_elements(By.TAG_NAME, 'button')
    for b in cookie_button:
        if b.text == 'Accetta tutto':
            b.click()
    return driver

def getPlayerAnag(driver, player_dict):
    try:
        more_button = driver.find_element(By.XPATH,'//*[@id="meta_more_button"]')
        more_button.click()
    except:
        pass

    #anag_div = driver.find_element(By.XPATH, '/html/body/div[4]/div[3]/div[1]/div[2]')
    #player = anag_div.find_element(By.TAG_NAME, 'h1').text
    i=1
    anag_elem1= ''

    try:
        driver.find_element(By.XPATH,f'//*[@id="meta"]/div[2]')
        tab_prefix = '//*[@id="meta"]/div[2]'
    except:
        tab_prefix = '//*[@id="meta"]/div'
    
    while not anag_elem1.startswith("Position"):
        anag_elem1 = driver.find_element(By.XPATH,f'{tab_prefix}/p[{i}]').text
        i+=1
        if i==10:
            raise KeyError
    #anag_elem1 = driver.find_element(By.XPATH,'//*[@id="meta"]/div[2]/p[1]').text
    anag_elem2 = driver.find_element(By.XPATH,f'{tab_prefix}/p[{i}]').text
    anag_elem3 = driver.find_element(By.XPATH,f'{tab_prefix}/p[{i+1}]').text
    anag_elem4 = driver.find_element(By.XPATH,f'{tab_prefix}/p[{i+2}]').text
    try:
        anag_elem5 = driver.find_element(By.XPATH,f'//*[@id="meta"]/div[2]/p[{i+3}]').text
    except:
        anag_elem5 = ''

    anag_elem1_split = anag_elem1.split('▪')
    anag_elem2_split = anag_elem2.split(' ')
    position = anag_elem1_split[0].split(':')[1].strip()
    #footed = anag_elem1_split[1].split(':')[1].strip()
    if anag_elem2.startswith('Born'):
        year_birth = anag_elem2.split(' ')[3]
        #height = ''
        #weight = ''
        nat = anag_elem3.split(' ')[2] if anag_elem3.startswith('National Team') else anag_elem3.split(' ')[1]
        #team = ' '.join(anag_elem5.split(' ')[1:]) if anag_elem4 != '' else ''

    else:
        #height, weight = anag_elem2_split[0].replace(',',''), anag_elem2_split[1]
        year_birth = anag_elem3.split(' ')[3]
        nat = anag_elem4.split(' ')[2] if anag_elem3.startswith('National Team') else anag_elem3.split(' ')[1]
        #team = ' '.join(anag_elem5.split(' ')[1:]) if anag_elem5 != '' else ''
        
    addict = dict(#player=player, 
                position=position 
                #foot=footed, 
                #height= height, 
                #weight=weight, 
                ,year_birth=year_birth
                ,nationality=nat
                #,team=team
                )

    return player_dict | addict


def getPlayerRecord(driver, player_dict):
    url = player_dict['link']
    driver.get(url)
    time.sleep(0.5)
    try:
        stats = getPlayerStats(driver)
        player_dict['stats'] = stats
    except:
        pass
    if 'stats' in player_dict.keys():
        player_dict = getPlayerAnag(driver, player_dict)
    
    
    return player_dict


def getTeamPlayers(driver, url : str, id_league: int, team: str) -> list:
    urls=[]
    driver.get(url)
    #id_league = '12229'
    #table = driver.find_element(By.XPATH, '//*[@id="stats_standard_11"]/tbody')
    table = driver.find_element(By.CLASS_NAME, 'stats_table')
    
    rows = table.find_elements(By.TAG_NAME, 'tr')
    for r in rows:
        presenze = '0'
        th = r.find_element(By.TAG_NAME, 'th')
        if th.get_attribute('csk') != None:
            tds = r.find_elements(By.TAG_NAME, 'td')
            for td in tds:
                if td.get_attribute('data-stat') == 'games':
                    presenze = td.text
            if int(presenze) > 5:
                player_name = th.find_element(By.TAG_NAME, 'a').get_attribute('href').split('/')[-1].replace('-',' ')
                #player_name = th.text
                id_player = th.get_attribute('data-append-csv')
                #print(th.text,th.get_attribute('data-append-csv'), presenze)
                url_p=f"https://fbref.com/en/players/{id_player}/scout/{id_league}/{player_name.replace(' ', '-')}-Scouting-Report"
                urls.append(dict(id=id_player, name=player_name, link =url_p, team=team))
    return urls

def getLeagueTeams(driver, url, id):
    driver.get(url)
    team_links=[]
    #table = driver.find_element(By.XPATH, '//*[@id="results2023-202491_overall"]/tbody')
    #table = driver.find_element(By.TAG_NAME, 'tbody')
    table = driver.find_element(By.CLASS_NAME, 'stats_table')
    table = table.find_element(By.TAG_NAME, 'tbody')
    #print(table.text)
    rows = table.find_elements(By.TAG_NAME, 'tr')
    for r in rows:
        l = r.find_element(By.TAG_NAME,'a')
        team = l.text
        link= l.get_property('href')
        team_links.append(dict(id=id, team=team, link=link))
    return team_links


In [296]:
prompts = {}
records = readJson('Dataset/Fbref/prova_records_perc_v3.json')
for p in records:
    player_name = p['player']
    prompt = f""""You are a professional football scout with expertise in analyzing players' technical and tactical characteristics. 
            I need you to generate a detailed report for a player, based on the provided list of statistics that describe their performance averaged per 90 minutes. 
            For each statistics, it is indicate value and percentile. Percentile is a value between 0 and 100. High value for percentile means that the player is good in that statistic.
            Percentile comparison is made between players of same role.
            Your task is to analyze this data and provide a report as follows:

            ### Input Data:
                - Player: {player_name}
                - Position: {p['position']}
                - Year birth: {p['year_birth']}
                - Height: {p['height']}
                - Weight: {p['weight']}
                - Statistics per 90 minutes: 
                {p['stats']}

            ### Output Format:
            Your report should be structured in the following way:
            **Player**: {player_name}
            **Strengths**: 
            Highlight the player's key strengths evident from their playing style.
            **Weaknesses**: 
            Point out areas where the player needs improvement.
            **Summary**:
            A brief summary of the player's overall performance.


            ### Notes for Analysis:
            - Use concise and professional language.
            - The report should be realistic for scouting purposes.
            - Do not generate code or class structures. Focus only on the football analysis.
            - The output must be in plain text, clearly formatted according to the structure above.
            - Do not write the name of the player 
            - Do not include statistics into report
            
            ###Generated Report:"""
    prompts[player_name] = {'prompt': prompt}

writeJson(prompts, 'Descriptions/prova_stats.json')

Il file non è stato trovato.


TypeError: 'NoneType' object is not iterable

In [326]:
prompts = {}
records = readJson('Dataset/Fbref/prova_records_perc_v2.json')
position_dict = readJson('Dataset/Fbref/position_mapping.json')
stats_dict = {'Shooting': ['Shooting'],'Offensive':[ 'Goal and Shot Creation', 'Possession','Passing', 'Pass Types'], 'Defensive':['Defense', 'Miscellaneous Stats']}
for p in records[:1]:
    player_name = p['player']
    prompts[player_name] = {'prompt':{}}
    roles = getRoles(p['position'])
    roles_verb = [position_dict[x.strip()] for x in roles]
    roles_str = ', '.join(roles_verb)
    for k, tab  in stats_dict.items():
        stat = {}
        for t in tab:
            stat = stat | p['stats'][t]
        prompt = f""""You are a professional soccer scout with expertise in analyzing players' technical and tactical characteristics. 
                I need you to generate a description for a player, based on the provided list of statistics that describe their performance averaged per match played. 
                For each statistics, it is indicate value and percentile. Percentile is a value between 0 and 100. High value for percentile means that the player is good in that statistic.
                Percentile comparison is made between players who play in same positions.
                Description should be concise, the length should be around 200 tokens.

                Your task is to analyze this data and provide a description as follows:

                ### Input Data:
                    - Position: {roles_str}
                    - Statistics averaged per match about {k}: 
                    {stat}

                ### Output:
                A brief summary of the player's {k} performance, highlighting player's key strengths and where player needs improvements


                ### Notes for Analysis (keep attention to these notes!!):
                - Use concise and professional language.
                - The description should be realistic for scouting purposes.
                - Do not generate code or class structures. Focus only on the soccer analysis.
                - The output must be in plain text, clearly formatted according to the structure above.
                - Do not include statistics into report
                - Provide description only about data in input.
                - Do not include supposition and consideration about future.
                - Descriptions should be related to the position of the player.
                - Write only description and not your toughts.
                
                ###Generated Report:"""
        
        prompts[player_name]['prompt'][k] = prompt.replace('<br>', ' ')

writeJson(prompts, 'Descriptions/prova_stats_v2.json')

In [ ]:
prompt = f""""You are a professional soccer scout with expertise in analyzing players' technical and tactical characteristics.  
        Generate a concise description (200-300 tokens) of a player's {k} performance based on the provided per-match statistics. 
        The analysis should:    
            - Highlight the player's **key strengths** (high percentiles).
            - Identify potential **areas for improvement** (low percentiles).
            - Be **role-specific**, considering the player's position.

        ## Input Format:
            - **Position:** Preferred positions 
            - **Statistics:** Per-match data with **values** and **percentiles** (0-100). A high percentile indicates **strong performance** relative to players in the same position.
        
        ## Output Guidelines:
            - **Professional & concise** language suitable for scouting.
            - **Plain text only** (no bullet points, code, or structured output).
            - **Do not include raw statistics** (focus on interpretation).
            - **No predictions** about future performance.
            - **Only use provided data**, without speculation.
            - **The description should be role-specific**, considering the player's position.
        
        ## Example Output:
            A well-rounded attacking midfielder with exceptional ability in progressing the ball and creating goal-scoring opportunities. He excels in shot-creating actions, with a strong ability to beat defenders through take-ons. His capacity to contribute directly to goals is elite. While highly effective in offensive play, his involvement in defensive phases and dead-ball situations is less prominent, indicating areas for potential development.

        ### Input Data:
            - **Position**: {roles_str}
            - **Statistics**: {stat}
            
        ###Generated Report:"""


In [365]:
leagues = readJson('Dataset/Fbref/competitions.json')
team_leagues = []
driver=initializeDriver()
for id, link in leagues.items():
    print(id,link)
    teams = getLeagueTeams(driver, link, id)
    team_leagues= team_leagues + teams

team_leagues

12192 https://fbref.com/en/comps/9/2023-2024/2023-2024-Premier-League-Stats
12202 https://fbref.com/en/comps/12/2023-2024/2023-2024-La-Liga-Stats
12207 https://fbref.com/en/comps/13/2023-2024/2023-2024-Ligue-1-Stats
12212 http://fbref.com/en/comps/20/2023-2024/2023-2024-Bundesliga-Stats
12229 https://fbref.com/en/comps/11/2023-2024/2023-2024-Serie-A-Stats


[{'id': '12192',
  'team': 'Manchester City',
  'link': 'https://fbref.com/en/squads/b8fd03ef/2023-2024/Manchester-City-Stats'},
 {'id': '12192',
  'team': 'Arsenal',
  'link': 'https://fbref.com/en/squads/18bb7c10/2023-2024/Arsenal-Stats'},
 {'id': '12192',
  'team': 'Liverpool',
  'link': 'https://fbref.com/en/squads/822bd0ba/2023-2024/Liverpool-Stats'},
 {'id': '12192',
  'team': 'Aston Villa',
  'link': 'https://fbref.com/en/squads/8602292d/2023-2024/Aston-Villa-Stats'},
 {'id': '12192',
  'team': 'Tottenham',
  'link': 'https://fbref.com/en/squads/361ca564/2023-2024/Tottenham-Hotspur-Stats'},
 {'id': '12192',
  'team': 'Chelsea',
  'link': 'https://fbref.com/en/squads/cff3d9bb/2023-2024/Chelsea-Stats'},
 {'id': '12192',
  'team': 'Newcastle Utd',
  'link': 'https://fbref.com/en/squads/b2b47a98/2023-2024/Newcastle-United-Stats'},
 {'id': '12192',
  'team': 'Manchester Utd',
  'link': 'https://fbref.com/en/squads/19538871/2023-2024/Manchester-United-Stats'},
 {'id': '12192',
  'team

In [368]:
#team_leagues[-20:]
writeJson(team_leagues, 'Dataset/Fbref/teams.json')

In [481]:
team_leagues = readJson('Dataset/Fbref/teams.json')
player_links= []
driver=initializeDriver()
with tqdm(total=len(team_leagues), desc="Extracting players link") as pbar:
    for team in team_leagues:
        players_team = getTeamPlayers(driver, team['link'], team['id'], team['team'])
        player_links= player_links + players_team
        #print(player_links)
        pbar.update(1)

driver.quit()
writeJson(player_links, 'Dataset/Fbref/players.json')

Extracting players link: 100%|██████████| 96/96 [33:23<00:00, 20.87s/it]  


In [ ]:
player_links[-1]

2309

In [461]:
players = readJson('Dataset/Fbref/players.json')
driver = initializeDriver()
with tqdm(total=len(players), desc="Processing players") as pbar:

    for i in range(len(players)):
        if 'stats' not in players[i].keys():
            players[i] = getPlayerRecord(driver, players[i])
        
        if i %10 == 0:
            writeJson(players, 'Dataset/Fbref/players.json')
        
        pbar.update(1)
writeJson(players, 'Dataset/Fbref/players.json')
driver.quit()

Processing players:  16%|█▌        | 368/2309 [22:33<1:58:58,  3.68s/it] 


NoSuchElementException: Message: no such element: Unable to locate element: {"method":"xpath","selector":"//*[@id="meta"]/div[2]/p[1]"}
  (Session info: chrome=132.0.6834.111); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#no-such-element-exception
Stacktrace:
	GetHandleVerifier [0x00007FF70EF502F5+28725]
	(No symbol) [0x00007FF70EEB2AE0]
	(No symbol) [0x00007FF70ED4510A]
	(No symbol) [0x00007FF70ED993D2]
	(No symbol) [0x00007FF70ED995FC]
	(No symbol) [0x00007FF70EDE3407]
	(No symbol) [0x00007FF70EDBFFEF]
	(No symbol) [0x00007FF70EDE0181]
	(No symbol) [0x00007FF70EDBFD53]
	(No symbol) [0x00007FF70ED8A0E3]
	(No symbol) [0x00007FF70ED8B471]
	GetHandleVerifier [0x00007FF70F27F30D+3366989]
	GetHandleVerifier [0x00007FF70F2912F0+3440688]
	GetHandleVerifier [0x00007FF70F2878FD+3401277]
	GetHandleVerifier [0x00007FF70F01AAAB+858091]
	(No symbol) [0x00007FF70EEBE74F]
	(No symbol) [0x00007FF70EEBA304]
	(No symbol) [0x00007

In [476]:
p = players[368]
getPlayerAnag(driver,p)


//*[@id="meta"]/div
Murillo
Murillo Santiago Costa dos Santos
Position: DF (CB, left)
180cm (5-11)
Born: July 4, 2002 (Age: 22-227d) in São Paulo, Brazil br
Citizenship: Brazil br
Club: Nottingham Forest


{'id': '1704b0b8',
 'name': 'Murillo',
 'link': 'https://fbref.com/en/players/1704b0b8/scout/12192/Murillo-Scouting-Report',
 'stats': {'Standard Stats': {'Goals': {'description': 'Goals scored or allowed',
    'value': '0.00',
    'percentile': '18'},
   'Assists': {'description': 'Assists', 'value': '0.06', 'percentile': '77'},
   'Goals + Assists': {'description': 'Goals and Assists',
    'value': '0.06',
    'percentile': '39'},
   'Non-Penalty Goals': {'description': 'Non-Penalty Goals',
    'value': '0.00',
    'percentile': '18'},
   'Penalty Kicks Made': {'description': 'Penalty Kicks Made',
    'value': '0.00',
    'percentile': '50'},
   'Penalty Kicks Attempted': {'description': 'Penalty Kicks Attempted',
    'value': '0.00',
    'percentile': '50'},
   'Yellow Cards': {'description': 'Yellow Cards',
    'value': '0.16',
    'percentile': '51'},
   'Red Cards': {'description': 'Red Cards',
    'value': '0.00',
    'percentile': '59'},
   'xG: Expected Goals': {'description':

In [475]:
def getPlayerAnag(driver, player_dict):
    try:
        more_button = driver.find_element(By.XPATH,'//*[@id="meta_more_button"]')
        more_button.click()
    except:
        pass

    #anag_div = driver.find_element(By.XPATH, '/html/body/div[4]/div[3]/div[1]/div[2]')
    #player = anag_div.find_element(By.TAG_NAME, 'h1').text
    i=1
    anag_elem1= ''

    try:
        driver.find_element(By.XPATH,f'//*[@id="meta"]/div[2]')
        tab_prefix = '//*[@id="meta"]/div[2]'
    except:
        tab_prefix = '//*[@id="meta"]/div'
    
    print(tab_prefix)
    print(driver.find_element(By.XPATH,tab_prefix).text)
    
    while not anag_elem1.startswith("Position"):
        anag_elem1 = driver.find_element(By.XPATH,f'{tab_prefix}/p[{i}]').text
        i+=1
        if i==10:
            raise KeyError
    #anag_elem1 = driver.find_element(By.XPATH,'//*[@id="meta"]/div[2]/p[1]').text
    anag_elem2 = driver.find_element(By.XPATH,f'{tab_prefix}/p[{i}]').text
    anag_elem3 = driver.find_element(By.XPATH,f'{tab_prefix}/p[{i+1}]').text
    anag_elem4 = driver.find_element(By.XPATH,f'{tab_prefix}/p[{i+2}]').text
    try:
        anag_elem5 = driver.find_element(By.XPATH,f'//*[@id="meta"]/div[2]/p[{i+3}]').text
    except:
        anag_elem5 = ''

    anag_elem1_split = anag_elem1.split('▪')
    anag_elem2_split = anag_elem2.split(' ')
    position = anag_elem1_split[0].split(':')[1].strip()
    #footed = anag_elem1_split[1].split(':')[1].strip()
    if anag_elem2.startswith('Born'):
        year_birth = anag_elem2.split(' ')[3]
        height = ''
        weight = ''
        nat = anag_elem3.split(' ')[2] if anag_elem3.startswith('National Team') else anag_elem3.split(' ')[1]
        team = ' '.join(anag_elem5.split(' ')[1:]) if anag_elem4 != '' else ''

    else:
        height, weight = anag_elem2_split[0].replace(',',''), anag_elem2_split[1]
        year_birth = anag_elem3.split(' ')[3]
        nat = anag_elem4.split(' ')[2] if anag_elem3.startswith('National Team') else anag_elem3.split(' ')[1]
        team = ' '.join(anag_elem5.split(' ')[1:]) if anag_elem5 != '' else ''
        
    addict = dict(#player=player, 
                position=position, 
                #foot=footed, 
                height= height, 
                weight=weight, 
                year_birth=year_birth, 
                nationality=nat, team=team)

    return player_dict | addict



In [478]:
players[359]

{'id': '2944f86f',
 'name': 'Frank Onyeka',
 'link': 'https://fbref.com/en/players/2944f86f/scout/12192/Frank-Onyeka-Scouting-Report',
 'stats': {'Standard Stats': {'Goals': {'description': 'Goals scored or allowed',
    'value': '0.08',
    'percentile': '49'},
   'Assists': {'description': 'Assists', 'value': '0.16', 'percentile': '71'},
   'Goals + Assists': {'description': 'Goals and Assists',
    'value': '0.23',
    'percentile': '62'},
   'Non-Penalty Goals': {'description': 'Non-Penalty Goals',
    'value': '0.08',
    'percentile': '49'},
   'Penalty Kicks Made': {'description': 'Penalty Kicks Made',
    'value': '0.00',
    'percentile': '45'},
   'Penalty Kicks Attempted': {'description': 'Penalty Kicks Attempted',
    'value': '0.00',
    'percentile': '45'},
   'Yellow Cards': {'description': 'Yellow Cards',
    'value': '0.62',
    'percentile': '1'},
   'Red Cards': {'description': 'Red Cards',
    'value': '0.00',
    'percentile': '58'},
   'xG: Expected Goals': {'desc